In [0]:
from pyspark.sql.functions import *

clean_customer = spark.table("default.clean_customer")
raw_orders = spark.table("default.raw_orders")
raw_order_details = spark.table("default.raw_order_details")
raw_products = spark.table("default.raw_products")

In [0]:
clean_customer.printSchema()
raw_orders.printSchema()
raw_order_details.printSchema()
raw_products.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Batch_ID: integer (nullable = true)

root
 |-- OrderID: string (nullable = true)
 |-- CustID: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Batch_ID: integer (nullable = true)

root
 |-- OrderID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Batch_ID: integer (nullable = true)

root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)


In [0]:
joined_df = raw_orders.alias("o") \
    .join(
        clean_customer.alias("c"),
        col("o.CustID") == col("c.Customer_ID"),
        "inner"
    ) \
    .join(
        raw_order_details.alias("d"),
        col("o.OrderID") == col("d.OrderID"),
        "inner"
    ) \
    .join(
        raw_products.alias("p"),
        col("d.ProductID") == col("p.ProductID"),
        "inner"
    )

In [0]:
joined_df.show(5)

+-------+------+------+----------+--------------------+-------------+--------+-----------+----------+-------------------+------+-------+--------------------+-------------+--------+-------+---------+--------+--------------------+----------------+--------+---------+-----------+----------+-----+--------------------+-------------+--------+
|OrderID|CustID|Amount| OrderDate|           Load_Date|Record_Source|Batch_ID|Customer_ID|      Name|              Email|  City|Country|           Load_Date|Record_Source|Batch_ID|OrderID|ProductID|Quantity|           Load_Date|   Record_Source|Batch_ID|ProductID|ProductName|  Category|Price|           Load_Date|Record_Source|Batch_ID|
+-------+------+------+----------+--------------------+-------------+--------+-----------+----------+-------------------+------+-------+--------------------+-------------+--------+-------+---------+--------+--------------------+----------------+--------+---------+-----------+----------+-----+--------------------+----------

In [0]:
curated_customer_orders = joined_df.select(

    col("o.OrderID").alias("Order_ID"),

    col("c.Customer_ID"),

    col("c.Name").alias("Customer_Name"),

    col("c.Email"),

    col("c.City"),

    col("p.ProductID"),

    col("p.ProductName").alias("Product_Name"),

    col("p.Category"),

    col("d.Quantity"),

    col("p.Price"),

    col("o.Amount").alias("Order_Amount"),

    col("o.OrderDate")

)

In [0]:
from pyspark.sql.functions import upper, col

curated_customer_orders = curated_customer_orders.withColumn(
    "Customer_Name",
    upper(col("Customer_Name"))
)

In [0]:
curated_customer_orders.select("Customer_Name").show(5, False)

+-------------+
|Customer_Name|
+-------------+
|CUSTOMER_1   |
|CUSTOMER_2   |
|CUSTOMER_3   |
|CUSTOMER_4   |
|CUSTOMER_5   |
+-------------+
only showing top 5 rows


In [0]:
curated_customer_orders = curated_customer_orders.fillna({
    "City": "Unknown",
    "Category": "Others"
})

In [0]:
curated_customer_orders.show(5)

+--------+-----------+-------------+-------------------+------+---------+------------+----------+--------+-----+------------+----------+
|Order_ID|Customer_ID|Customer_Name|              Email|  City|ProductID|Product_Name|  Category|Quantity|Price|Order_Amount| OrderDate|
+--------+-----------+-------------+-------------------+------+---------+------------+----------+--------+-----+------------+----------+
|    O100|       C001|   CUSTOMER_1|customer1@gmail.com|City_2|     P001|   Product_1|Category_2|       1|   60|       100.0|2024-01-01|
|    O101|       C002|   CUSTOMER_2|customer2@gmail.com|City_3|     P002|   Product_2|Category_3|       2|   70|       123.5|2024-01-02|
|    O102|       C003|   CUSTOMER_3|customer3@gmail.com|City_4|     P003|   Product_3|Category_4|       3|   80|       147.0|2024-01-03|
|    O103|       C004|   CUSTOMER_4|customer4@gmail.com|City_5|     P004|   Product_4|Category_5|       4|   90|       170.5|2024-01-04|
|    O104|       C005|   CUSTOMER_5|custo

In [0]:
%sql
DROP TABLE IF EXISTS default.curated_customer_orders;

In [0]:
curated_customer_orders.write.mode("overwrite").saveAsTable(
    "default.curated_customer_orders"
)

In [0]:
%sql
SELECT COUNT(*) FROM default.curated_customer_orders;

COUNT(*)
500


In [0]:
%sql
SELECT * 
FROM default.curated_customer_orders
LIMIT 20;

Order_ID,Customer_ID,Customer_Name,Email,City,ProductID,Product_Name,Category,Quantity,Price,Order_Amount,OrderDate
O100,C001,CUSTOMER_1,customer1@gmail.com,City_2,P001,Product_1,Category_2,1,60,100.0,2024-01-01
O101,C002,CUSTOMER_2,customer2@gmail.com,City_3,P002,Product_2,Category_3,2,70,123.5,2024-01-02
O102,C003,CUSTOMER_3,customer3@gmail.com,City_4,P003,Product_3,Category_4,3,80,147.0,2024-01-03
O103,C004,CUSTOMER_4,customer4@gmail.com,City_5,P004,Product_4,Category_5,4,90,170.5,2024-01-04
O104,C005,CUSTOMER_5,customer5@gmail.com,City_6,P005,Product_5,Category_1,5,100,194.0,2024-01-05
O105,C006,CUSTOMER_6,customer6@gmail.com,City_7,P006,Product_6,Category_2,1,110,217.5,2024-01-06
O106,C007,CUSTOMER_7,customer7@gmail.com,City_8,P007,Product_7,Category_3,2,120,241.0,2024-01-07
O107,C008,CUSTOMER_8,customer8@gmail.com,City_9,P008,Product_8,Category_4,3,130,264.5,2024-01-08
O108,C009,CUSTOMER_9,customer9@gmail.com,City_10,P009,Product_9,Category_5,4,140,288.0,2024-01-09
O109,C010,CUSTOMER_10,customer10@gmail.com,City_1,P010,Product_10,Category_1,5,150,311.5,2024-01-10
